In [1]:
import os
import json
import traceback
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

from agent_01 import save_company_pdf, ask_company
from agent_02 import plan_creative_strategy
from agent_03 import write_creative_scripts, load_scripts
from agent_04 import create_poster
from agent_05 import send_creative_email
from agent_06 import create_campaign_video

d:\AI & Agentic Enginnering\agentic ai week7\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\AI & Agentic Enginnering\agentic ai week7\.venv\Lib\site-packages\fpdf\__init__.py:43: UserWarning: You have both PyFPDF & fpdf2 installed. Both packages cannot be installed at the same time as they share the same module namespace. To only keep fpdf2, run: pip uninstall --yes pypdf && pip install --upgrade fpdf2
  warnings.warn(


In [2]:
MODEL = "gpt-5.4-nano" 
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
openai = OpenAI(api_key=openai_api_key)

In [3]:
# walkflow

system_message = """
You are the Planner of a creative digital marketing agency. You manage a team of agents. Each agent is one of your tools:
 - save_company_pdf (RAG Agent): saves a client's company PDF into our database and analyzes it.
 - ask_company (RAG Agent): answers questions about a saved client.
 - plan_creative_strategy (Strategy Agent): researches trends and plans the best concept.
 - write_creative_scripts (Creative Writer Agent): writes the professional campaign script and its PDF.
 - create_poster (Poster Create Agent): designs the poster image.
 - create_campaign_video (Animation Agent): creates the 20 second vertical video from the scenes and poster.
 - send_to_client (Product Manager Agent): writes hook + caption and emails the poster, PDF and video.

How to work:
 1. When the user uploads a PDF, ALWAYS call save_company_pdf first, using the exact path shown in
    "[User uploaded a PDF: ...]". If you don't know the company name, ask for it first.
 2. When the user asks for a campaign, run the agents in this exact order, one at a time:
    plan_creative_strategy -> write_creative_scripts -> create_poster -> create_campaign_video -> send_to_client
 3. Before create_campaign_video, tell the user it can take several minutes.
 4. If the user gives an email address, pass it to send_to_client. Otherwise leave it empty.
 5. If a tool says there is no data for the company, ask the user to upload their company PDF.
 6. If a tool returns an error, tell the user which agent failed and stop.
 Keep replies short: tell the user what each agent did and list the files created.
"""

In [4]:
save_company_pdf_tool = {
    "name": "save_company_pdf",
    "description": "Save a client's PDF document into the knowledge base and rebuild the vector database. Call this whenever the user uploads or attaches a PDF for a company. Must be called before asking anything about a new client.",
    "parameters": {
        "type": "object",
        "properties": {
            "pdf_path": {"type": "string", "description": "The file path of the uploaded PDF"},
            "company_name": {"type": "string", "description": "The client company name, e.g. Sunny Lane Cafe"},
        },
        "required": ["pdf_path", "company_name"],
        "additionalProperties": False,
    },
}

ask_company_tool = {
    "name": "ask_company",
    "description": "Answer a question about a client using only that client's own documents in the knowledge base. Use it when the user asks what the company does, who their audience is, their brand voice, colours or services.",
    "parameters": {
        "type": "object",
        "properties": {
            "company_name": {"type": "string", "description": "The client company name"},
            "question": {"type": "string", "description": "The question to answer from the client's documents"},
        },
        "required": ["company_name", "question"],
        "additionalProperties": False,
    },
}

plan_creative_strategy_tool = {
    "name": "plan_creative_strategy",
    "description": "STEP 1 of a campaign. The strategy agent reads the client's documents and searches the web for current trends, then recommends one poster concept. Returns the full strategy text. Always call this first when the user asks for a campaign, poster or marketing idea.",
    "parameters": {
        "type": "object",
        "properties": {
            "company_name": {"type": "string", "description": "The client company name"},
            "goal": {"type": "string", "description": "What the campaign must achieve, in the user's own words"},
        },
        "required": ["company_name", "goal"],
        "additionalProperties": False,
    },
}

write_creative_scripts_tool = {
    "name": "write_creative_scripts",
    "description": "STEP 2. The creative writer turns the strategy into a professional campaign script (overview, poster script, 5-scene storyboard, brand notes) and saves it as JSON and PDF. Call only after plan_creative_strategy.",
    "parameters": {"type": "object", "properties": {}, "additionalProperties": False},
}

create_poster_tool = {
    "name": "create_poster",
"description": "STEP 3. The poster create agent generates the poster image from the script and saves it as a PNG. Call only after write_creative_scripts.",    "parameters": {"type": "object", "properties": {}, "additionalProperties": False},
}

create_campaign_video_tool = {
    "name": "create_campaign_video",
    "description": "STEP 4. The animation agent creates a 20 second vertical MP4: animated AI scenes, the poster with a slow zoom, and a call-to-action end card. Takes several minutes. Call only after create_poster.",
    "parameters": {"type": "object", "properties": {}, "additionalProperties": False},
}

send_to_client_tool = {
    "name": "send_to_client",
    "description": "STEP 5, final. The product manager writes the social media hook and caption and emails the poster (PNG), script (PDF) and video (MP4) to the client. Call only after create_campaign_video.",
    "parameters": {
        "type": "object",
        "properties": {
            "client_email": {"type": "string", "description": "Client email address if the user gave one; otherwise leave empty"},
        },
        "additionalProperties": False,
    },
}

In [5]:
tools = [
    {"type": "function", "function": save_company_pdf_tool},
    {"type": "function", "function": ask_company_tool},
    {"type": "function", "function": plan_creative_strategy_tool},
    {"type": "function", "function": write_creative_scripts_tool},
    {"type": "function", "function": create_poster_tool},
    {"type": "function", "function": create_campaign_video_tool},
    {"type": "function", "function": send_to_client_tool},
]

In [6]:
campaign = {}

def handle_tool_calls(tool_calls):
    responses = []
    for tool_call in tool_calls:
        name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        print(f"PLANNER: calling {name} {args}", flush=True)

        try:
            if name == "save_company_pdf":
                result = save_company_pdf(args["pdf_path"], args["company_name"])

            elif name == "ask_company":
                result = ask_company(args["company_name"], args["question"])

            elif name == "plan_creative_strategy":
                campaign.clear()                     # new campaign: forget the old files
                campaign["company_name"] = args["company_name"]
                campaign["strategy"] = plan_creative_strategy(args["company_name"], args["goal"])
                result = campaign["strategy"]

            elif name == "write_creative_scripts":
                campaign["scripts_path"], campaign["pdf_path"] = write_creative_scripts(
                    campaign["company_name"], campaign["strategy"])
                script = load_scripts(campaign["scripts_path"])
                result = (f"Script ready: {campaign['pdf_path']}\n"
                          f"Campaign: {script['campaign_overview']['campaign_name']}\n"
                          f"Headline: {script['poster']['headline']}")

            elif name == "create_poster":
                campaign["poster_path"] = create_poster(campaign["company_name"], campaign["scripts_path"])
                result = f"Poster saved: {campaign['poster_path']}"

            elif name == "create_campaign_video":
                campaign["video_path"] = create_campaign_video(
                    campaign["company_name"], campaign["scripts_path"], campaign["poster_path"])
                result = f"Video saved: {campaign['video_path']}"

            elif name == "send_to_client":
                result = send_creative_email(
                    campaign["company_name"], campaign["scripts_path"], campaign["poster_path"],
                    campaign["pdf_path"], campaign.get("video_path"), args.get("client_email") or None)

            else:
                result = f"Unknown tool: {name}"

        except KeyError as e:
            result = f"Error in {name}: missing {e}. An earlier step has not run yet."
        except Exception as e:
            traceback.print_exc()
            result = f"Error in {name}: {type(e).__name__}: {e}"

        responses.append({"role": "tool", "content": str(result), "tool_call_id": tool_call.id})
    return responses

In [7]:
def text_of(content):
    # Gradio stores text as a string, but file messages in different shapes depending on version
    if isinstance(content, str):
        return content
    if isinstance(content, (tuple, list)) and content and isinstance(content[0], str):
        return f"[User uploaded a PDF: {content[0]}]"
    if isinstance(content, dict):
        return f"[User uploaded a PDF: {content.get('path', '')}]"

    parts = []
    for c in content:
        if isinstance(c, str):
            parts.append(c)
        elif c.get("type") == "text":
            parts.append(c["text"])
        elif c.get("type") == "file":
            parts.append(f"[User uploaded a PDF: {c['file']['path']}]")
    return "\n".join(parts)

def chat(message, history):
    user_text = message["text"]
    for file in message["files"]:
        path = file if isinstance(file, str) else file.get("path", "")
        user_text += f"\n[User uploaded a PDF: {path}]"

    cleaned_history = [{"role": h["role"], "content": text_of(h["content"])} for h in history]
    messages = [{"role": "system", "content": system_message}] + cleaned_history + [{"role": "user", "content": user_text}]

    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,
        parallel_tool_calls=False,
    )
    while response.choices[0].finish_reason == "tool_calls":
        tool_message = response.choices[0].message
        tool_responses = handle_tool_calls(tool_message.tool_calls)
        messages.append(tool_message)
        messages.extend(tool_responses)
        response = openai.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            parallel_tool_calls=False,
        )

    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(
    fn=chat,
    multimodal=True,
    title="Creative Digital Marketing Agency"
).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


PLANNER: calling save_company_pdf {'pdf_path': 'C:\\Users\\Asus\\AppData\\Local\\Temp\\gradio\\57622ad7e92cbae784f82f44de294a9847c4cce6f028999fb11134374d73a6f5\\matcha_lane_cafe_company_profile.pdf', 'company_name': 'Matcha Lane Café'}
RAG AGENT: saving C:\Users\Asus\AppData\Local\Temp\gradio\57622ad7e92cbae784f82f44de294a9847c4cce6f028999fb11134374d73a6f5\matcha_lane_cafe_company_profile.pdf for Matcha Lane Café
RAG AGENT: vector db built with 17 chunks
RAG AGENT: asking about Matcha Lane Café: Give a short profile of this company: what they sell, target audience, brand voice, brand colours, unique selling points and any marketing goals mentioned.
PLANNER: calling plan_creative_strategy {'company_name': 'Matcha Lane Café', 'goal': 'Promote their Sakura Season menu and increase weekday afternoon visits (2 PM–5 PM) by 25% during December.'}
STRATEGY AGENT: Planning for Matcha Lane Café - Promote their Sakura Season menu and increase weekday afternoon visits (2 PM–5 PM) by 25% during Dec

MoviePy - Done !
MoviePy - video ready outputs\matcha_lane_café_video_20261123_201915.mp4
ANIMATION AGENT: saved outputs\matcha_lane_café_video_20261123_201915.mp4 (20s, 8.4 MB)
PLANNER: calling send_to_client {'client_email': 'amandakarunathilaka490@gmail.com'}
PRODUCT MANAGER AGENT: sending campaign for Matcha Lane Café to amandakarunathilaka490@gmail.com
PRODUCT MANAGER AGENT: Email sent to amandakarunathilaka490@gmail.com with 3 attachments (10.2 MB).
